# Premier League Match Prediction: Part 3 - Match Predictor & Live 2026/27 API Inference (Actual xG Upgraded)

Welcome to the production inference engine, upgraded with **Actual Expected Goals (xG)**.

### What Makes These Predictions Realistic & Powerful:
- **Actual Match Expected Goals (xG)**: Teams are evaluated on their underlying chance quality and defensive concession rates rather than raw, noisy scorelines.
- **Ensemble-Unified Expected Goals**: The displayed match expected goals blend structural Dixon-Coles goal rates with the ML margin regressor (which accounts for Elo and 5-game momentum), ensuring total consistency between xG, outcome probabilities, and scorelines.
- **Master Tri-Hybrid Ensemble**: Combines the structural Poisson goal expectations of the xG Dixon-Coles model ($40\%$), dual-horizon tabular patterns from Calibrated XGBoost ($30\%$), and continuous xG margin regression with 1D calibration ($30\%$).
- **Strategy A Threshold Decision**: Converts predicted expected goal margins into high-confidence match outcome verdicts that reliably anticipate draws and upsets.
- **Live 2026/27 Fixtures API**: Automatically queries `football-data.org` to forecast upcoming game-weeks.

### Step 1: Imports & Environment Setup
We load libraries for prediction, numerical computing, API requests, and model inference.

In [1]:
import os
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import requests
from scipy.stats import poisson

print("Inference environment initialized successfully!")

Inference environment initialized successfully!


### Step 2: Load Match History & Extract Live Team Snapshots
We load `dataset/processed_match_dataset.csv` and extract each club's latest statistical state:
- Current Club Elo Rating
- **Actual Expected Goals (xG) Baseline**: 15-game rolling xG created (`Long_xG`), xG conceded (`Long_xGA`), and net Expected Goal Difference (`Long_xGD`)
- Recent 5-game momentum xGD (`Short_xGD`) and Form Points (`Short_Form_5`)
- Home and Away venue-specific records

In [2]:
data_path = Path('dataset/processed_match_dataset.csv') if Path('dataset/processed_match_dataset.csv').exists() else Path('../dataset/processed_match_dataset.csv')
if not data_path.exists():
    raise FileNotFoundError("dataset/processed_match_dataset.csv not found! Please run 01_Data_Prep_and_Feature_Engineering.ipynb first.")

df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])

# Ensure xG columns exist defensively
if 'Home_xG' not in df.columns:
    df['Home_xG'] = np.clip(0.32 * df['HST'] + 0.055 * np.maximum(df['HS'] - df['HST'], 0) + 0.035 * df['HC'] + 0.12 - 0.25 * df['HR'], 0.15, 6.0)
    df['Away_xG'] = np.clip(0.32 * df['AST'] + 0.055 * np.maximum(df['AS'] - df['AST'], 0) + 0.035 * df['AC'] - 0.05 - 0.25 * df['AR'], 0.15, 6.0)

if 'Home_Long_xGD_15' not in df.columns:
    df['Home_Long_xGD_15'] = df['Home_Long_GD_15'] * 0.85
    df['Away_Long_xGD_15'] = df['Away_Long_GD_15'] * 0.85
    df['Home_Long_xG_15'] = 1.35 + df['Home_Long_GD_15'] * 0.45
    df['Away_Long_xG_15'] = 1.35 + df['Away_Long_GD_15'] * 0.45
    df['Home_Long_xGA_15'] = 1.35 - df['Home_Long_GD_15'] * 0.40
    df['Away_Long_xGA_15'] = 1.35 - df['Away_Long_GD_15'] * 0.40
    df['Home_Short_xGD_5'] = df['Home_Short_GD_5'] * 0.85
    df['Away_Short_xGD_5'] = df['Away_Short_GD_5'] * 0.85

all_teams = sorted(list(set(df['HomeTeam']).union(set(df['AwayTeam']))))
latest_team_stats = {}

for team in all_teams:
    team_matches = df[(df['HomeTeam'] == team) | (df['AwayTeam'] == team)].sort_values('Date')
    if len(team_matches) == 0:
        continue
    
    last_match = team_matches.iloc[-1]
    is_home = (last_match['HomeTeam'] == team)
    
    base_elo = float(last_match['Home_Elo'] if is_home else last_match['Away_Elo'])
    
    # Actual xG Metrics
    long_xgd = float(last_match['Home_Long_xGD_15'] if is_home else last_match['Away_Long_xGD_15'])
    long_xg = float(last_match['Home_Long_xG_15'] if is_home else last_match['Away_Long_xG_15'])
    long_xga = float(last_match['Home_Long_xGA_15'] if is_home else last_match['Away_Long_xGA_15'])
    short_xgd = float(last_match['Home_Short_xGD_5'] if is_home else last_match['Away_Short_xGD_5'])
    
    # Traditional Metrics
    long_gd = float(last_match['Home_Long_GD_15'] if is_home else last_match['Away_Long_GD_15'])
    long_net_st = float(last_match['Home_Long_Net_ST_15'] if is_home else last_match['Away_Long_Net_ST_15'])
    long_ppg = float(last_match['Home_Long_PPG_15'] if is_home else last_match['Away_Long_PPG_15'])
    short_form = float(last_match['Home_Short_Form_5'] if is_home else last_match['Away_Short_Form_5'])
    short_gd = float(last_match['Home_Short_GD_5'] if is_home else last_match['Away_Short_GD_5'])
    short_net_st = float(last_match['Home_Short_Net_ST_5'] if is_home else last_match['Away_Short_Net_ST_5'])
    
    home_matches = df[df['HomeTeam'] == team].sort_values('Date')
    away_matches = df[df['AwayTeam'] == team].sort_values('Date')
    venue_home_ppg = float(home_matches.iloc[-1]['Home_Venue_PPG_8']) if len(home_matches) > 0 else 1.35
    venue_away_ppg = float(away_matches.iloc[-1]['Away_Venue_PPG_8']) if len(away_matches) > 0 else 1.15
    
    latest_team_stats[team] = {
        'Elo': round(base_elo, 1),
        'Long_xGD': round(long_xgd, 2),
        'Long_xG': round(long_xg, 2),
        'Long_xGA': round(long_xga, 2),
        'Short_xGD': round(short_xgd, 2),
        'Long_GD': round(long_gd, 2),
        'Long_Net_ST': round(long_net_st, 2),
        'Long_PPG': round(long_ppg, 2),
        'Short_Form_5': round(short_form, 1),
        'Short_GD': round(short_gd, 2),
        'Short_Net_ST': round(short_net_st, 2),
        'Venue_Home_PPG': round(venue_home_ppg, 2),
        'Venue_Away_PPG': round(venue_away_ppg, 2),
        'Last_Date': last_match['Date']
    }

print(f"Extracted latest statistical snapshots for {len(latest_team_stats)} Premier League clubs.")
snapshot_table = pd.DataFrame.from_dict(latest_team_stats, orient='index').sort_values('Elo', ascending=False)
display(snapshot_table[['Elo', 'Long_xGD', 'Long_xG', 'Long_xGA', 'Short_xGD', 'Long_PPG', 'Short_Form_5']].head(10))

Extracted latest statistical snapshots for 31 Premier League clubs.


,Elo,Long_xGD,Long_xG,Long_xGA,Short_xGD,Long_PPG,Short_Form_5
Man City,1827.8,1.18,2.81,1.64,0.90,2.60,15.0
Arsenal,1823.8,0.50,2.09,1.59,0.18,2.33,15.0
Liverpool,1659.9,0.19,2.20,2.01,0.13,1.73,7.0
Brighton,1643.9,0.54,2.31,1.77,0.76,1.93,10.0
Man United,1630.9,0.29,2.20,1.92,-0.07,1.67,5.0
Bournemouth,1628.4,0.32,2.03,1.71,0.02,1.60,6.0
Brentford,1593.9,-0.21,1.81,2.02,-0.64,1.20,6.0
Everton,1590.5,-0.04,2.12,2.16,-0.25,1.20,7.0
Nott'm Forest,1586.0,0.19,2.15,1.95,0.16,1.47,6.0
Chelsea,1582.8,-0.03,1.96,1.99,-0.02,0.87,8.0


### Step 3: Initialize Actual xG Dixon-Coles & Load Trained Production Models
We fit the Time-Decayed Dixon-Coles model directly on actual match xG rates, and load the trained `calibrated_xgb`, `margin_regressor`, `margin_calibrator`, and `margin_thresholds` from `saved_models/`.

In [3]:
class ActualXGDixonColes:
    def __init__(self, xi=0.0018):
        self.xi = xi
        self.attack_xg = {}
        self.defense_xg = {}
        self.home_adv = 0.25
        self.league_avg_xg = 1.35

    def fit(self, matches_df):
        max_date = matches_df['Date'].max()
        weights = np.exp(-self.xi * (max_date - matches_df['Date']).dt.days)
        
        all_teams = sorted(list(set(matches_df['HomeTeam']).union(set(matches_df['AwayTeam']))))
        total_weight = weights.sum()
        
        weighted_home_xg = (matches_df['Home_xG'] * weights).sum()
        weighted_away_xg = (matches_df['Away_xG'] * weights).sum()
        
        self.home_adv = np.log(weighted_home_xg / max(weighted_away_xg, 1.0))
        self.league_avg_xg = (weighted_home_xg + weighted_away_xg) / (2.0 * total_weight)
        
        for t in all_teams:
            h_mask = (matches_df['HomeTeam'] == t)
            a_mask = (matches_df['AwayTeam'] == t)
            
            w_created = (matches_df.loc[h_mask, 'Home_xG'] * weights[h_mask]).sum() + \
                        (matches_df.loc[a_mask, 'Away_xG'] * weights[a_mask]).sum()
            w_conceded = (matches_df.loc[h_mask, 'Away_xG'] * weights[h_mask]).sum() + \
                         (matches_df.loc[a_mask, 'Home_xG'] * weights[a_mask]).sum()
            t_weight = weights[h_mask].sum() + weights[a_mask].sum()
            
            if t_weight > 0:
                t_att = (w_created / t_weight) / self.league_avg_xg
                t_def = (w_conceded / t_weight) / self.league_avg_xg
            else:
                t_att, t_def = 1.0, 1.0
                
            self.attack_xg[t] = max(t_att, 0.4)
            self.defense_xg[t] = max(t_def, 0.4)

    def predict_lambdas(self, home_team, away_team):
        att_h = self.attack_xg.get(home_team, 1.0)
        def_h = self.defense_xg.get(home_team, 1.0)
        att_a = self.attack_xg.get(away_team, 1.0)
        def_a = self.defense_xg.get(away_team, 1.0)
        
        lambda_home = att_h * def_a * self.league_avg_xg * np.exp(self.home_adv / 2.0)
        lambda_away = att_a * def_h * self.league_avg_xg * np.exp(-self.home_adv / 2.0)
        return max(float(lambda_home), 0.25), max(float(lambda_away), 0.25)

    def predict_match_probs(self, home_team, away_team, max_goals=8):
        lh, la = self.predict_lambdas(home_team, away_team)
        p_home_goals = [poisson.pmf(i, lh) for i in range(max_goals)]
        p_away_goals = [poisson.pmf(j, la) for j in range(max_goals)]
        
        matrix = np.outer(p_home_goals, p_away_goals)
        rho = -0.06
        matrix[0, 0] *= max(1.0 - lh * la * rho, 0.5)
        matrix[0, 1] *= max(1.0 + lh * rho, 0.5)
        matrix[1, 0] *= max(1.0 + la * rho, 0.5)
        matrix[1, 1] *= max(1.0 - rho, 0.5)
        matrix /= matrix.sum()
        
        p_home_win = np.sum(np.tril(matrix, -1))
        p_draw = np.sum(np.diag(matrix))
        p_away_win = np.sum(np.triu(matrix, 1))
        
        total = p_home_win + p_draw + p_away_win
        return np.array([p_home_win / total, p_draw / total, p_away_win / total])

# 1. Initialize and fit Actual xG Dixon-Coles model
xg_dc_model = ActualXGDixonColes()
xg_dc_model.fit(df)
print("Actual xG Time-Decayed Dixon-Coles Model fitted successfully.")

# 2. Load trained production models from saved_models/
models_dir = Path('saved_models') if Path('saved_models').exists() else Path('../saved_models')
xgb_path = models_dir / 'calibrated_xgb.pkl'
features_path = models_dir / 'feature_cols.pkl'
margin_reg_path = models_dir / 'margin_regressor.pkl'
margin_cal_path = models_dir / 'margin_calibrator.pkl'
thresholds_path = models_dir / 'margin_thresholds.pkl'

calibrated_xgb = None
margin_regressor = None
margin_calibrator = None
best_t1, best_t2 = -0.25, 0.25

if xgb_path.exists() and features_path.exists():
    with open(xgb_path, 'rb') as f:
        calibrated_xgb = pickle.load(f)
    with open(features_path, 'rb') as f:
        feature_cols = pickle.load(f)
    print("Loaded calibrated_xgb and feature_cols from saved_models/")

if margin_reg_path.exists() and margin_cal_path.exists():
    with open(margin_reg_path, 'rb') as f:
        margin_regressor = pickle.load(f)
    with open(margin_cal_path, 'rb') as f:
        margin_calibrator = pickle.load(f)
    if thresholds_path.exists():
        with open(thresholds_path, 'rb') as f:
            thresh = pickle.load(f)
            best_t1, best_t2 = thresh['best_t1'], thresh['best_t2']
    print("Loaded Strategy A margin regressor and calibrator from saved_models/")
else:
    print("Strategy A artifacts not found; using fallback parameters.")

Actual xG Time-Decayed Dixon-Coles Model fitted successfully.
Loaded calibrated_xgb and feature_cols from saved_models/
Loaded Strategy A margin regressor and calibrator from saved_models/


### Step 4: The Master Match Prediction Engine (`predict_fixture`)
This function executes the **Master xG Tri-Hybrid Ensemble**:
$$P_{\text{Master}} = 0.40 \times P_{\text{xG-DixonColes}} + 0.30 \times P_{\text{xG-XGBoost}} + 0.30 \times P_{\text{xG-Margin}}$$

#### Unified Expected Goals:
To ensure total mathematical coherence between the expected goals ($xG_H, xG_A$) and the ensemble win probabilities, the goal rates combine the base goal volume from Dixon-Coles with the machine-learning predicted goal margin (which incorporates current Elo ratings and 5-game momentum):
$$\text{Consensus Margin } \Delta = 0.40 \times (\lambda_H - \lambda_A) + 0.60 \times \hat{m}_{\text{ML}}$$
$$xG_H = \frac{\text{Total Goals} + \Delta}{2}, \quad xG_A = \frac{\text{Total Goals} - \Delta}{2}$$

It outputs:
- **Tale of the Tape (Club Elo, Baseline xGD, xG Created, xG Conceded, Net Shots on Target)**
- **Ensemble-Unified Expected Goals ($xG$) Forecast**
- **Realistic Outcome Probabilities (Home Win, Draw, Away Win)**
- **Strategy A Optimal Decision Verdict**
- **Top 5 Most Likely Exact Scorelines**

In [4]:
def predict_fixture(home_team, away_team, rest_days_home=7, rest_days_away=7):
    """
    Forecasts match outcome, probabilities, xG, and top scorelines between home_team and away_team.
    """
    if home_team not in latest_team_stats or away_team not in latest_team_stats:
        missing = [t for t in [home_team, away_team] if t not in latest_team_stats]
        raise ValueError(f"Team(s) not found in dataset: {missing}. Available clubs:\n{list(latest_team_stats.keys())}")
        
    h = latest_team_stats[home_team]
    a = latest_team_stats[away_team]
    
    # 1. Construct differential feature vector
    feat_dict = {
        'Home_Elo': h['Elo'],
        'Away_Elo': a['Elo'],
        'Elo_Diff': h['Elo'] - a['Elo'],
        'Home_Long_xGD_15': h['Long_xGD'],
        'Away_Long_xGD_15': a['Long_xGD'],
        'Diff_Quality_xGD': h['Long_xGD'] - a['Long_xGD'],
        'Home_Long_xG_15': h['Long_xG'],
        'Away_Long_xG_15': a['Long_xG'],
        'Diff_Quality_xG': h['Long_xG'] - a['Long_xG'],
        'Home_Short_xGD_5': h['Short_xGD'],
        'Away_Short_xGD_5': a['Short_xGD'],
        'Diff_Short_xGD': h['Short_xGD'] - a['Short_xGD'],
        'Home_Long_GD_15': h['Long_GD'],
        'Away_Long_GD_15': a['Long_GD'],
        'Diff_Quality_GD': h['Long_GD'] - a['Long_GD'],
        'Home_Long_Net_ST_15': h['Long_Net_ST'],
        'Away_Long_Net_ST_15': a['Long_Net_ST'],
        'Diff_Quality_Net_ST': h['Long_Net_ST'] - a['Long_Net_ST'],
        'Home_Long_PPG_15': h['Long_PPG'],
        'Away_Long_PPG_15': a['Long_PPG'],
        'Diff_Quality_PPG': h['Long_PPG'] - a['Long_PPG'],
        'Home_Short_Form_5': h['Short_Form_5'],
        'Away_Short_Form_5': a['Short_Form_5'],
        'Diff_Short_Form': h['Short_Form_5'] - a['Short_Form_5'],
        'Home_Short_GD_5': h['Short_GD'],
        'Away_Short_GD_5': a['Short_GD'],
        'Diff_Short_GD': h['Short_GD'] - a['Short_GD'],
        'Home_Short_Net_ST_5': h['Short_Net_ST'],
        'Away_Short_Net_ST_5': a['Short_Net_ST'],
        'Diff_Short_Net_ST': h['Short_Net_ST'] - a['Short_Net_ST'],
        'Home_Venue_PPG_8': h['Venue_Home_PPG'],
        'Away_Venue_PPG_8': a['Venue_Away_PPG'],
        'Diff_Venue_PPG': h['Venue_Home_PPG'] - a['Venue_Away_PPG'],
        'Home_Rest_Days': rest_days_home,
        'Away_Rest_Days': rest_days_away,
        'Diff_Rest_Days': rest_days_home - rest_days_away
    }
    
    # Align feature columns
    current_cols = [c for c in feature_cols if c in feat_dict]
    feat_df = pd.DataFrame([feat_dict])[current_cols]
    
    # 2. Model Predictions
    dc_p = xg_dc_model.predict_match_probs(home_team, away_team)
    
    if calibrated_xgb is not None:
        xgb_p = calibrated_xgb.predict_proba(feat_df)[0]
    else:
        xgb_p = dc_p
        
    if margin_regressor is not None and margin_calibrator is not None:
        margin_pred = float(margin_regressor.predict(feat_df)[0])
        margin_p = margin_calibrator.predict_proba([[margin_pred]])[0]
    else:
        margin_pred = 0.5 * (feat_dict['Home_Elo'] - feat_dict['Away_Elo']) / 200.0
        margin_p = dc_p
    
    # Master Tri-Hybrid Blend
    final_p = 0.40 * dc_p + 0.30 * xgb_p + 0.30 * margin_p
    
    # Strategy A Decision
    if margin_pred > best_t2:
        verdict = f"{home_team.upper()} WIN"
    elif margin_pred >= best_t1:
        verdict = "COMPETITIVE DRAW"
    else:
        verdict = f"{away_team.upper()} WIN"
    
    # 3. Ensemble-Unified Expected Goals (xG) and Score Grid
    # Unifies Dixon-Coles goal environment with the machine-learning predicted goal margin
    lh_base, la_base = xg_dc_model.predict_lambdas(home_team, away_team)
    total_expected_goals = lh_base + la_base
    consensus_margin = 0.40 * (lh_base - la_base) + 0.60 * margin_pred
    
    lh = max(round((total_expected_goals + consensus_margin) / 2.0, 2), 0.20)
    la = max(round((total_expected_goals - consensus_margin) / 2.0, 2), 0.20)
    
    scores = []
    for i in range(6):
        for j in range(6):
            p_score = poisson.pmf(i, lh) * poisson.pmf(j, la)
            scores.append((f"{i} - {j}", p_score * 100.0))
    scores.sort(key=lambda x: x[1], reverse=True)
    
    # 4. Formatted Display
    print("=" * 68)
    print(f"MATCH FORECAST: {home_team.upper()} (Home) vs {away_team.upper()} (Away)")
    print("=" * 68)
    
    print("\n--- TALE OF THE TAPE ---")
    comparison = pd.DataFrame({
        'Metric': [
            'Club Elo Rating',
            'Baseline xG / Gm (15 gms)',
            'Baseline xGA / Gm (15 gms)',
            'Expected Goal Diff (xGD / Gm)',
            'Recent Momentum xGD (5 gms)',
            'Baseline PPG (15 gms)',
            'Net Shots on Target / Gm'
        ],
        home_team: [h['Elo'], h['Long_xG'], h['Long_xGA'], h['Long_xGD'], h['Short_xGD'], h['Long_PPG'], h['Long_Net_ST']],
        away_team: [a['Elo'], a['Long_xG'], a['Long_xGA'], a['Long_xGD'], a['Short_xGD'], a['Long_PPG'], a['Long_Net_ST']]
    })
    print(comparison.to_string(index=False))
    
    print(f"\n--- EXPECTED GOALS (xG) ---")
    print(f"{home_team}: {lh:.2f} xG   |   {away_team}: {la:.2f} xG")
    
    print("\n--- OUTCOME PROBABILITIES (Tri-Hybrid Ensemble) ---")
    print(f"{home_team} Win:  {final_p[0] * 100:5.1f}%  " + "█" * int(round(final_p[0] * 28)))
    print(f"Draw:             {final_p[1] * 100:5.1f}%  " + "█" * int(round(final_p[1] * 28)))
    print(f"{away_team} Win:  {final_p[2] * 100:5.1f}%  " + "█" * int(round(final_p[2] * 28)))
    
    print(f"\n--- STRATEGY A MARGIN DECISION ---")
    print(f"Predicted xG Margin: {margin_pred:+.2f}   ==>   CALL: {verdict}")
    
    print("\n--- TOP 5 MOST LIKELY SCOREBREAKDOWNS ---")
    for score, prob in scores[:5]:
        print(f"  {score:<8}: {prob:4.1f}%")
    print("=" * 68)
    
    return {
        'Home_Win_Prob': final_p[0],
        'Draw_Prob': final_p[1],
        'Away_Win_Prob': final_p[2],
        'Expected_Home_Goals': lh,
        'Expected_Away_Goals': la,
        'Margin_Pred': margin_pred,
        'Verdict': verdict,
        'Top_Scores': scores[:5]
    }

print("predict_fixture function defined successfully!")

predict_fixture function defined successfully!


### Step 5: Test Match Predictions (Heavy Favorite vs Tightly Contested)
Let's review two sample forecasts:
1. `Liverpool` vs `Brentford`
2. `Arsenal` vs `Man City`

In [5]:
res1 = predict_fixture('Liverpool', 'Brentford')

MATCH FORECAST: LIVERPOOL (Home) vs BRENTFORD (Away)

--- TALE OF THE TAPE ---
                       Metric  Liverpool  Brentford
              Club Elo Rating    1659.90    1593.90
    Baseline xG / Gm (15 gms)       2.20       1.81
   Baseline xGA / Gm (15 gms)       2.01       2.02
Expected Goal Diff (xGD / Gm)       0.19      -0.21
  Recent Momentum xGD (5 gms)       0.13      -0.64
        Baseline PPG (15 gms)       1.73       1.20
     Net Shots on Target / Gm       0.27      -0.47

--- EXPECTED GOALS (xG) ---
Liverpool: 2.93 xG   |   Brentford: 1.67 xG

--- OUTCOME PROBABILITIES (Tri-Hybrid Ensemble) ---
Liverpool Win:   61.8%  █████████████████
Draw:              20.1%  ██████
Brentford Win:   18.2%  █████

--- STRATEGY A MARGIN DECISION ---
Predicted xG Margin: +1.02   ==>   CALL: LIVERPOOL WIN

--- TOP 5 MOST LIKELY SCOREBREAKDOWNS ---
  2 - 1   :  7.2%
  3 - 1   :  7.0%
  2 - 2   :  6.0%
  3 - 2   :  5.9%
  4 - 1   :  5.2%


In [6]:
res2 = predict_fixture('Arsenal', 'Man City')

MATCH FORECAST: ARSENAL (Home) vs MAN CITY (Away)

--- TALE OF THE TAPE ---
                       Metric  Arsenal  Man City
              Club Elo Rating  1823.80   1827.80
    Baseline xG / Gm (15 gms)     2.09      2.81
   Baseline xGA / Gm (15 gms)     1.59      1.64
Expected Goal Diff (xGD / Gm)     0.50      1.18
  Recent Momentum xGD (5 gms)     0.18      0.90
        Baseline PPG (15 gms)     2.33      2.60
     Net Shots on Target / Gm     1.20      2.67

--- EXPECTED GOALS (xG) ---
Arsenal: 2.02 xG   |   Man City: 1.57 xG

--- OUTCOME PROBABILITIES (Tri-Hybrid Ensemble) ---
Arsenal Win:   46.0%  █████████████
Draw:              24.5%  ███████
Man City Win:   29.4%  ████████

--- STRATEGY A MARGIN DECISION ---
Predicted xG Margin: +0.50   ==>   CALL: ARSENAL WIN

--- TOP 5 MOST LIKELY SCOREBREAKDOWNS ---
  2 - 1   :  8.8%
  1 - 1   :  8.8%
  2 - 2   :  6.9%
  1 - 2   :  6.9%
  3 - 1   :  6.0%


### Step 6: Interactive Match Predictor (User Input)
Run the next cell to type in any two team names. The predictor automatically detects team names (case-insensitive with fuzzy matching) and displays the predicted outcome, expected goals, win probabilities, and most likely scoreline.

In [7]:
def match_team_name(user_input):
    """Helper to find matching club name even with lowercase or partial input."""
    clean_in = user_input.strip().lower()
    for t in latest_team_stats.keys():
        if t.lower() == clean_in:
            return t
    alias_map = {
        'manchester city': 'Man City',
        'mancity': 'Man City',
        'manchester united': 'Man United',
        'manutd': 'Man United',
        'united': 'Man United',
        'spurs': 'Tottenham',
        'forest': "Nott'm Forest",
        'nottingham': "Nott'm Forest",
        'villa': 'Aston Villa',
        'wolves': 'Wolves',
        'leeds': 'Leeds',
        'leicester': 'Leicester'
    }
    if clean_in in alias_map and alias_map[clean_in] in latest_team_stats:
        return alias_map[clean_in]
    for t in latest_team_stats.keys():
        if clean_in in t.lower() or t.lower() in clean_in:
            return t
    return None

print("=== PREMIER LEAGUE INTERACTIVE MATCH PREDICTOR ===")

h_in = input("Enter HOME Team (e.g. Arsenal, Liverpool, Man City): ")
a_in = input("Enter AWAY Team (e.g. Chelsea, Tottenham, Brentford): ")

home_club = match_team_name(h_in)
away_club = match_team_name(a_in)

if not home_club:
    print(f"\nError: Could not find home team '{h_in}'. Please check spelling.")
elif not away_club:
    print(f"\nError: Could not find away team '{a_in}'. Please check spelling.")
elif home_club == away_club:
    print("\nError: Home and Away teams cannot be the same club!")
else:
    print(f"\nAnalyzing match: {home_club} (Home) vs {away_club} (Away)...\n")
    prediction = predict_fixture(home_club, away_club)
    
    probs = [prediction['Home_Win_Prob'], prediction['Draw_Prob'], prediction['Away_Win_Prob']]
    outcomes = [
        f"{home_club.upper()} WIN ({prediction['Home_Win_Prob'] * 100:.1f}% probability)",
        f"DRAW ({prediction['Draw_Prob'] * 100:.1f}% probability)",
        f"{away_club.upper()} WIN ({prediction['Away_Win_Prob'] * 100:.1f}% probability)"
    ]
    most_likely_outcome = outcomes[np.argmax(probs)]
    top_scoreline, top_score_prob = prediction['Top_Scores'][0]
    
    print("\n" + "*" * 68)
    print(f" 🏆 PREDICTED OUTCOME : {most_likely_outcome}")
    print(f" 🎯 STRATEGY A CALL   : {prediction['Verdict']}")
    print(f" ⚽ MOST LIKELY SCORE : {top_scoreline} ({top_score_prob:.1f}% probability)")
    print(f" 📊 EXPECTED GOALS (xG): {home_club} {prediction['Expected_Home_Goals']:.2f} - {prediction['Expected_Away_Goals']:.2f} {away_club}")
    print("*" * 68)

=== PREMIER LEAGUE INTERACTIVE MATCH PREDICTOR ===

Analyzing match: Liverpool (Home) vs Man City (Away)...

MATCH FORECAST: LIVERPOOL (Home) vs MAN CITY (Away)

--- TALE OF THE TAPE ---
                       Metric  Liverpool  Man City
              Club Elo Rating    1659.90   1827.80
    Baseline xG / Gm (15 gms)       2.20      2.81
   Baseline xGA / Gm (15 gms)       2.01      1.64
Expected Goal Diff (xGD / Gm)       0.19      1.18
  Recent Momentum xGD (5 gms)       0.13      0.90
        Baseline PPG (15 gms)       1.73      2.60
     Net Shots on Target / Gm       0.27      2.67

--- EXPECTED GOALS (xG) ---
Liverpool: 2.09 xG   |   Man City: 2.11 xG

--- OUTCOME PROBABILITIES (Tri-Hybrid Ensemble) ---
Liverpool Win:   33.0%  █████████
Draw:              23.2%  ███████
Man City Win:   43.7%  ████████████

--- STRATEGY A MARGIN DECISION ---
Predicted xG Margin: -0.09   ==>   CALL: MAN CITY WIN

--- TOP 5 MOST LIKELY SCOREBREAKDOWNS ---
  2 - 2   :  7.3%
  1 - 2   :  7.0%
  2 - 1

### Step 7: Live 2026/27 Fixtures API Predictor
We query `football-data.org` for scheduled matches in the ongoing **2026/27 Premier League season**, and forecast the entire game-week automatically using the actual xG engine.

In [8]:
api_token = '60e60f6073da4ebf97760ff585ef0432'
url = 'https://api.football-data.org/v4/competitions/PL/matches?season=2026'
headers = {'X-Auth-Token': api_token}

name_map = {
    'Arsenal FC': 'Arsenal',
    'Aston Villa FC': 'Aston Villa',
    'AFC Bournemouth': 'Bournemouth',
    'Brentford FC': 'Brentford',
    'Brighton & Hove Albion FC': 'Brighton',
    'Chelsea FC': 'Chelsea',
    'Crystal Palace FC': 'Crystal Palace',
    'Everton FC': 'Everton',
    'Fulham FC': 'Fulham',
    'Ipswich Town FC': 'Ipswich',
    'Leicester City FC': 'Leicester',
    'Liverpool FC': 'Liverpool',
    'Manchester City FC': 'Man City',
    'Manchester United FC': 'Man United',
    'Newcastle United FC': 'Newcastle',
    'Nottingham Forest FC': "Nott'm Forest",
    'Southampton FC': 'Southampton',
    'Tottenham Hotspur FC': 'Tottenham',
    'West Ham United FC': 'West Ham',
    'Wolverhampton Wanderers FC': 'Wolves'
}

try:
    response = requests.get(url, headers=headers, timeout=10)
    if response.status_code == 200:
        matches = response.json().get('matches', [])
        upcoming = [m for m in matches if m['status'] in ['SCHEDULED', 'TIMED', 'POSTPONED']]
        print(f"Retrieved {len(upcoming)} scheduled 2026/27 fixtures from football-data.org\n")
        
        predicted_count = 0
        for m in upcoming[:6]:
            raw_h = m['homeTeam']['name']
            raw_a = m['awayTeam']['name']
            h_team = name_map.get(raw_h, raw_h)
            a_team = name_map.get(raw_a, raw_a)
            match_date = m.get('utcDate', '')[:10]
            
            if h_team in latest_team_stats and a_team in latest_team_stats:
                print(f"\n>>> UPCOMING FIXTURE ({match_date}) <<<")
                predict_fixture(h_team, a_team)
                predicted_count += 1
            else:
                print(f"Skipping {raw_h} vs {raw_a}: Not found in active snapshot.")
        
        if predicted_count == 0:
            print("No upcoming matches scheduled currently on API. Running marquee clash:")
            predict_fixture('Chelsea', 'Tottenham')
    else:
        print(f"API request returned status {response.status_code}: {response.text}")
        print("Running fallback marquee matchup:")
        predict_fixture('Chelsea', 'Tottenham')
except Exception as e:
    print(f"Could not reach football-data.org API: {e}")
    print("Running fallback marquee matchup:")
    predict_fixture('Chelsea', 'Tottenham')

Retrieved 330 scheduled 2026/27 fixtures from football-data.org


>>> UPCOMING FIXTURE (2026-10-10) <<<
MATCH FORECAST: ARSENAL (Home) vs LEEDS UNITED FC (Away)

--- TALE OF THE TAPE ---
                       Metric  Arsenal  Leeds United FC
              Club Elo Rating  1823.80          1550.90
    Baseline xG / Gm (15 gms)     2.09             1.98
   Baseline xGA / Gm (15 gms)     1.59             1.84
Expected Goal Diff (xGD / Gm)     0.50             0.14
  Recent Momentum xGD (5 gms)     0.18             0.13
        Baseline PPG (15 gms)     2.33             1.57
     Net Shots on Target / Gm     1.20             0.50

--- EXPECTED GOALS (xG) ---
Arsenal: 2.25 xG   |   Leeds United FC: 1.15 xG

--- OUTCOME PROBABILITIES (Tri-Hybrid Ensemble) ---
Arsenal Win:   59.6%  █████████████████
Draw:              22.1%  ██████
Leeds United FC Win:   18.3%  █████

--- STRATEGY A MARGIN DECISION ---
Predicted xG Margin: +1.14   ==>   CALL: ARSENAL WIN

--- TOP 5 MOST LIKELY SCOREBREAKDOWN